In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.features.batted_ball import batted_ball_events, add_quality_flags

df = load_all_snapshots()
f = add_discipline_flags(df)
f = f[f["pitch_type"].notna()]

outcomes = (
    f.groupby("pitch_type")
    .agg(
        pitches=("is_swing", "size"),
        zone_pct=("in_zone", "mean"),
        swing_pct=("is_swing", "mean"),
    )
)
outcomes["whiff_pct"] = f[f["is_swing"]].groupby("pitch_type")["is_whiff"].mean()

# chase: swings out of zone / pitches out of zone
oz = f[~f["in_zone"]]
outcomes["chase_pct"] = oz.groupby("pitch_type")["is_swing"].mean()

outcomes = outcomes[outcomes["pitches"] >= 2000]
print(outcomes.round(3).sort_values("whiff_pct", ascending=False).to_string())

            pitches  zone_pct  swing_pct  whiff_pct  chase_pct
pitch_type                                                    
KC            12349     0.410      0.441      0.329      0.322
FS            21872     0.372      0.522      0.327      0.359
SL           104113     0.453      0.488      0.323      0.321
ST            51922     0.442      0.456      0.297      0.308
CU            43648     0.436      0.415      0.296      0.290
CH            72389     0.390      0.503      0.293      0.337
SV             3695     0.431      0.412      0.271      0.292
FC            58083     0.516      0.490      0.213      0.272
FF           225518     0.554      0.489      0.189      0.239
SI           111694     0.568      0.457      0.117      0.246


In [2]:
bbe = add_quality_flags(batted_ball_events(df))
bbe = bbe[bbe["pitch_type"].notna()].copy()

la = pd.to_numeric(bbe["launch_angle"], errors="coerce")
bbe["is_ground_ball"] = la < 10          # MLB glossary: GB is under 10 deg
bbe["is_popup"] = la > 50                # over 50 deg

contact = (
    bbe.groupby("pitch_type")
    .agg(
        bbe=("is_barrel", "size"),
        gb_pct=("is_ground_ball", "mean"),
        popup_pct=("is_popup", "mean"),
        barrel_pct=("is_barrel", "mean"),
        hard_hit_pct=("is_hard_hit", "mean"),
        avg_ev=("launch_speed", "mean"),
        xwoba=("estimated_woba_using_speedangle", "mean"),
    )
)

full = outcomes.join(contact)
full = full[full["bbe"] >= 500]
print(full.round(3).sort_values("xwoba").to_string())

            pitches  zone_pct  swing_pct  whiff_pct  chase_pct    bbe  gb_pct  popup_pct  barrel_pct  hard_hit_pct  avg_ev  xwoba
pitch_type                                                                                                                       
ST            51922     0.442      0.456      0.297      0.308   8328   0.355      0.134       0.072         0.293  85.324  0.344
CH            72389     0.390      0.503      0.293      0.337  14031   0.511      0.075       0.061          0.32   85.87  0.347
FS            21872     0.372      0.522      0.327      0.359   3771   0.554      0.054       0.057         0.353  86.671  0.349
SL           104113     0.453      0.488      0.323      0.321  17177   0.433      0.099       0.072         0.348  86.932  0.361
SV             3695     0.431      0.412      0.271      0.292    585   0.422      0.099       0.075         0.325  87.251  0.363
SI           111694     0.568      0.457      0.117      0.246  23634    0.57      0.052  

In [3]:
print(full.loc[["SI", "FF", "SL", "CH"],
               ["whiff_pct", "gb_pct", "barrel_pct", "hard_hit_pct", "xwoba"]]
      .round(3).to_string())

            whiff_pct  gb_pct  barrel_pct  hard_hit_pct  xwoba
pitch_type                                                    
SI              0.117    0.57       0.066         0.435  0.368
FF              0.189   0.346         0.1         0.447  0.392
SL              0.323   0.433       0.072         0.348  0.361
CH              0.293   0.511       0.061          0.32  0.347


In [4]:
z = (full[["whiff_pct", "chase_pct", "gb_pct", "popup_pct"]]
     - full[["whiff_pct", "chase_pct", "gb_pct", "popup_pct"]].mean()
     ) / full[["whiff_pct", "chase_pct", "gb_pct", "popup_pct"]].std()

z["primary_weapon"] = z.idxmax(axis=1)
print(z.round(2).to_string())

            whiff_pct  chase_pct  gb_pct  popup_pct primary_weapon
pitch_type                                                        
CH               0.39       0.99    0.72      -0.41      chase_pct
CU               0.43      -0.24    -0.0       -0.6      whiff_pct
FC              -0.74      -0.69   -0.59       0.52      popup_pct
FF              -1.09      -1.54   -1.42       1.54      popup_pct
FS               0.87       1.58    1.28       -1.1      chase_pct
KC               0.91       0.60    0.53      -0.96      whiff_pct
SI              -2.11      -1.35    1.49      -1.15         gb_pct
SL               0.82       0.59   -0.28       0.34      whiff_pct
ST               0.45       0.23    -1.3       1.46      popup_pct
SV               0.08      -0.16   -0.43       0.35      popup_pct


In [5]:
from src.features.arsenal import build_arsenal
from src.data.player_ids import load_player_ids, display_name

# whiff율만으로 평가한 싱커볼러 vs 전체
ars = build_arsenal(f)
si_heavy = ars.xs("SI", level="pitch_type")
si_heavy = si_heavy[si_heavy["usage"] >= 0.35]

sw = f[f["is_swing"]]
pitcher_whiff = sw.groupby("pitcher")["is_whiff"].agg(["mean", "size"])
pitcher_whiff = pitcher_whiff[pitcher_whiff["size"] >= 200]

pbbe = add_quality_flags(batted_ball_events(df))
pla = pd.to_numeric(pbbe["launch_angle"], errors="coerce")
pbbe["is_gb"] = pla < 10
pitcher_gb = pbbe.groupby("pitcher").agg(gb=("is_gb", "mean"), n=("is_gb", "size"))
pitcher_gb = pitcher_gb[pitcher_gb["n"] >= 100]

comp = pitcher_whiff.join(pitcher_gb, how="inner")
comp["si_heavy"] = comp.index.isin(si_heavy.index)

print(comp.groupby("si_heavy")[["mean", "gb"]].agg(["mean", "size"]).round(3).to_string())

           mean          gb     
           mean size   mean size
si_heavy                        
False     0.236  367  0.425  367
True      0.216   78  0.521   78
